# Paired SNARE-seq integration: Chen et al. 2019

The RNA and ATAC cells are paired, but the tutorial deliberately follows the same gene-activity label-transfer path used for unpaired data. No intermediate `.h5ad` caches are required.

- RNA: [Chen-2019-RNA.h5ad](https://ndownloader.figshare.com/files/59742638)
- ATAC: [Chen-2019-ATAC.h5ad](https://ndownloader.figshare.com/files/59742644)
- Gene annotation: [GENCODE mouse vM25](https://ndownloader.figshare.com/files/59759393)
- Original study: [GEO GSE126074](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE126074)


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
rna_path = fetch("Chen-2019-RNA.h5ad", "https://ndownloader.figshare.com/files/59742638")
atac_path = fetch("Chen-2019-ATAC.h5ad", "https://ndownloader.figshare.com/files/59742644")
gtf_path = fetch("gencode.vM25.chr_patch_hapl_scaff.annotation.gtf.gz", "https://ndownloader.figshare.com/files/59759393")


## Build gene activity directly from the published inputs


In [ ]:
import anndata as ad
rna = sc.read_h5ad(rna_path)
atac = sc.read_h5ad(atac_path)
max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", "8000"))
if max_cells > 0:
    rna = subsample(rna, max_cells)
    atac = subsample(atac, max_cells)
rna.layers["counts"] = rna.X.copy()
ga = scb.pp.create_gene_activity(atac, rna, gtf_file=str(gtf_path), verbose=True)
ga.layers["ga_smooth"] = ga.layers["ga_smooth"] if "ga_smooth" in ga.layers else ga.X.copy()
ga.obs["cell_type"] = atac.obs["cell_type"].astype(str).to_numpy()
rna.shape, ga.shape


## Autoencoder map, optimal transport, and supBIOT


In [ ]:
rna.obs_names = "rna:" + rna.obs_names.astype(str)
ga.obs_names = "atac:" + ga.obs_names.astype(str)
rna.obs["cell_type"] = rna.obs["cell_type"].astype(str)
rna.obs["true_cell_type"] = rna.obs["cell_type"].astype(str)
ga.obs["true_cell_type"] = ga.obs["cell_type"].astype(str)
adata = scb.pp.autoencoder_map(
    rna, ga, label="modality", keys=("reference", "query"),
    reference_layer="counts", query_layer="ga_smooth",
    label_key="cell_type", unlabeled_category="Unknown", out_key="X_ae",
    n_top_genes=3000, latent_dim=30, max_epochs=AE_EPOCHS,
    early_stop_patience=5, random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata, obsm_key="X_ae", batch_key="modality", out_key="X_supbiot",
    label_key="cell_type", unlabeled_category="Unknown",
    prealign="ot", prealign_strength=0.8, align_reference=True,
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
adata = scb.ot.supbiot(
    adata, use_rep="X_supbiot", input_rep_key="X_ae",
    label_key="cell_type", unlabeled_category="Unknown",
    pred_label_key="pred_cell_type", pred_conf_key="pred_confidence",
    min_conf=0.0, random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


In [ ]:
query = adata[adata.obs["modality"].astype(str).eq("query")].copy()
sc.pp.neighbors(adata, use_rep="X_supbiot", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["modality", "cell_type", "pred_cell_type"])
query.obs["pred_cell_type"].value_counts().head(10)
